# Membersihkan Baris Duplikat pada Dataset GPS

Notebook ini menggunakan kecepatan **DuckDB** untuk membaca dataset `.parquet` lengkap, menghapus baris yang terduplikasi secara persis (_exact duplicates_), dan menyimpannya kembali ke file `.parquet` yang baru (agar file asli tetap utuh).

In [6]:
import duckdb
import time
import os

# Konfigurasi Path File Parquet (sesuaikan bila berbeda)
input_file = "./DataGPS_parquet/all_gps_data.parquet"
output_file = "./DataGPS_parquet/all_gps_data_no_dup.parquet"  # Hasil disimpan dengan nama _clean

# Validasi kemunculan file
if not os.path.exists(input_file):
    print(f"Error: File {input_file} tidak ditemukan. Silakan cek path input!")
else:
    print(f"File Input 📁  : {input_file}")
    print(f"File Output 📁 : {output_file}")

File Input 📁  : ./DataGPS_parquet/all_gps_data.parquet
File Output 📁 : ./DataGPS_parquet/all_gps_data_no_dup.parquet


In [7]:
# 1. Konfigurasi DuckDB menggunakan shared.py
from shared import get_con
con = get_con(os.getcwd())

print(f"Mengeksekusi proses penghapusan duplikasi... (Proses ini bisa memakan waktu mulai dari beberapa detik hingga menit)")
start_time = time.time()

# 2. Query ekspor `.parquet` yang baru dari source dengan distinct (unik) record order by untuk memastikan urutan data tetap terjaga
query_export = f"""
    COPY (
        SELECT DISTINCT * 
        FROM read_parquet('{input_file}')
        ORDER BY maid, timestamp
    ) TO '{output_file}' (FORMAT 'parquet', COMPRESSION 'zstd');
"""

# Menjalankan eksekusi query (langsung export menjadi _clean)
con.execute(query_export)

elapsed = time.time() - start_time
print(f"Selesai menyalin ke file baru dalam {elapsed:.2f} detik!")

Mengeksekusi proses penghapusan duplikasi... (Proses ini bisa memakan waktu mulai dari beberapa detik hingga menit)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Selesai menyalin ke file baru dalam 391.29 detik!


In [11]:
print("Memverifikasi dan membandingkan file...")
con = get_con(os.getcwd())

# Menghitung baris sebelum dan sesudah
rows_before = con.execute(f"SELECT COUNT(*) FROM read_parquet('{input_file}')").fetchone()[0]
rows_after  = con.execute(f"SELECT COUNT(*) FROM read_parquet('{output_file}')").fetchone()[0]

print("="*60)
print(f"📊 Jumlah baris SEBELUM dibersihkan : {rows_before:,}")
print(f"📊 Jumlah baris SESUDAH dibersihkan : {rows_after:,}")
print(f"🔥 Total baris duplikat dihapus     : {rows_before - rows_after:,}")
print("="*60)

Memverifikasi dan membandingkan file...
📊 Jumlah baris SEBELUM dibersihkan : 310,445,351
📊 Jumlah baris SESUDAH dibersihkan : 169,509,324
🔥 Total baris duplikat dihapus     : 140,936,027


In [12]:
schema_out = con.execute(f"""
    SELECT column_name, column_type 
    FROM (DESCRIBE SELECT * FROM read_parquet('{output_file}'))
""").df()

print(f"\n  Schema output:")
for _, row in schema_out.iterrows():
    print(f"    {row['column_name']:<15} {row['column_type']}")

# Verifikasi sorting
sort_check = con.execute(f"""
    WITH numbered AS (
        SELECT maid, timestamp,
                LAG(maid) OVER () AS prev_maid,
                LAG(timestamp) OVER () AS prev_ts
        FROM read_parquet('{output_file}')
        LIMIT 1000000
    )
    SELECT COUNT(*) AS violations
    FROM numbered
    WHERE prev_maid IS NOT NULL
        AND (maid < prev_maid 
            OR (maid = prev_maid AND timestamp < prev_ts))
""").fetchone()[0]

print(f"\n  🔀 Sort violations (1M baris pertama): {sort_check:,}")
print(f"  {'✅' if sort_check == 0 else '⚠️'} Data {'tersortir dengan benar' if sort_check == 0 else 'TIDAK tersortir!'}")

con.close()



  Schema output:
    maid            VARCHAR
    latitude        DOUBLE
    longitude       DOUBLE
    timestamp       BIGINT

  🔀 Sort violations (1M baris pertama): 0
  ✅ Data tersortir dengan benar
